
# Импорт библиотек


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# Пути к данным


In [5]:
case_df = pd.read_parquet("../data/case_7_data_for_rd.snappy.parquet")

truth_df = pd.read_parquet("../data/truth_rd_data.snappy.parquet")

definitions_df = pd.read_excel("../data/TG_Definitions.xlsx")

In [6]:
case_df.head(10)

,rd_documentnumber,rank,rd_data,tg_ids
0,,1,"{""number"": """", ""product"": {""productName"": """", ...",[]
1,,2,"{""number"": """", ""product"": {""productName"": """", ...",[]
2,AM.01.01.01.003.R.000013.01.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[35]
3,AM.01.01.01.003.R.000014.01.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
4,AM.01.01.01.003.R.000018.07.20,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
5,AM.01.01.01.003.R.000021.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
6,AM.01.01.01.003.R.000022.02.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
7,AM.01.01.01.003.R.000023.02.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[45]
8,AM.01.01.01.003.R.000025.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....","[45, 37]"
9,AM.01.01.01.003.R.000029.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[45]


In [7]:
from pprint import pprint
pprint(case_df.loc[2, "rd_data"])

('{"type": 13, "active": true, "number": "AM.01.01.01.003.R.000013.01.22", '
 '"docNorm": "спецификация изготовителя. ", "useArea": "Для реализации '
 'населению в качестве биологически активной добавки к пище - дополнительного '
 'источника витаминов D3, В12, источника куркумина, содержащей йод. ", '
 '"nameProd": "Биологически активная добавка к пище \\"КурКумин с витаминами '
 'D3 и B12\\" (\\"CoreCumin + Vitamins D3 & B12\\"), жидкость во флаконах по '
 '5-100 мл с дозатором-капельницей или дозатором пипеткой. ", "protocol": '
 '"Взамен свидетельства о государственной регистрации '
 'AM.01.01.01.003.R.000274.10.21 от 19.10.2021 г, экспертного заключения № '
 'ЭЗ/224 от 11.10.2021 г., протокола испытаний № 1515 от 08.10.2021 г., '
 'выданных санитарно-гигиенической испытательной лабораторией ЗАО '
 '«Национальный институт здравоохранения имени академика С. Авдалбекяна» МЗ РА '
 '(аттестат аккредитации № 015/Т-081 до 14.10.2022 г.).", "firmGetName": "ООО '
 '\\"Альфа-Консалтинг\\"", 

Правило №1: tg_ids никогда не используется в качестве признака модели.(используем только для анализа)

In [8]:
type(case_df.loc[2, "tg_ids"])

numpy.ndarray

In [9]:
case_df["rank"].value_counts()

rank
1    3595803
2       7287
3        409
4         18
Name: count, dtype: int64

In [10]:
truth_df.shape #(573354, 5)
#truth_df.head()
print(truth_df.loc[2, 'rd_number'])
truth_df.iloc[0]


Условия хранения стандартные для данного вида продукции, в соответствии со статьей 3 ТР ТС 009/2011 «О безопасности парфюмерно-косметической продукции». Срок и условия хранения (годности), эксплуатации указан в прилагаемой к продукции товаросопроводительной документации и/или на упаковке и/или каждой единице продукции. Декларация выдана взамен ЕАЭС N RU Д-TR.РА02.В.66396/22


tg                                                             4
rd_type                                                      N/A
group_tnved                                                 3303
code_tnved                                            3303009000
rd_number      Декларация о соответствии распространяется на ...
Name: 0, dtype: str

In [11]:
truth_df.head()

,tg,rd_type,group_tnved,code_tnved,rd_number
0,4,N/A,3303,3303009000,Декларация о соответствии распространяется на ...
1,4,N/A,3303,3303001000,ГОСТ 31678-2012 Продукция парфюмерная жидкая. ...
2,4,N/A,3303,3303009000,Условия хранения стандартные для данного вида ...
3,4,N/A,3303,3303001000,Дата изготовления отобранных образцов (проб) п...
4,4,N/A,3303,3303001000,ГОСТ 32893-2014 Продукция парфюмерно-косметиче...


1) Большинство документов относятся к одной товарной группе, однако существует заметное количество документов с несколькими товарными группами. Возможно, задача имеет элементы multi-label классификации, хотя целевой Truth содержит только три интересующие нас группы.

In [12]:
case_df["tg_ids"].apply(len).value_counts().sort_index()

tg_ids
0         50
1    2685718
2     763343
3     132771
4      17344
5       3025
6        998
7        248
8         20
Name: count, dtype: int64

In [13]:
case_df["rank"].value_counts().sort_index()

rank
1    3595803
2       7287
3        409
4         18
Name: count, dtype: int64

In [14]:
import json

sample = json.loads(case_df.loc[2, "rd_data"])
sample.keys()

dict_keys(['type', 'active', 'number', 'docNorm', 'useArea', 'nameProd', 'protocol', 'firmGetName', 'statusGroup', 'firmMadeName', 'activeFromDate'])

In [15]:
for key in sample:
    print(f"{key}: {sample[key]}")

type: 13
active: True
number: AM.01.01.01.003.R.000013.01.22
docNorm: спецификация изготовителя. 
useArea: Для реализации населению в качестве биологически активной добавки к пище - дополнительного источника витаминов D3, В12, источника куркумина, содержащей йод. 
nameProd: Биологически активная добавка к пище "КурКумин с витаминами D3 и B12" ("CoreCumin + Vitamins D3 & B12"), жидкость во флаконах по 5-100 мл с дозатором-капельницей или дозатором пипеткой. 
protocol: Взамен свидетельства о государственной регистрации AM.01.01.01.003.R.000274.10.21 от 19.10.2021 г, экспертного заключения № ЭЗ/224 от 11.10.2021 г., протокола испытаний № 1515 от 08.10.2021 г., выданных санитарно-гигиенической испытательной лабораторией ЗАО «Национальный институт здравоохранения имени академика С. Авдалбекяна» МЗ РА (аттестат аккредитации № 015/Т-081 до 14.10.2022 г.).
firmGetName: ООО "Альфа-Консалтинг"
statusGroup: 1
firmMadeName: «Nurish.Me, Inc.»
activeFromDate: 2022-01-25


In [16]:
truth_df["tg"].value_counts()

tg
35    349165
43    223640
4        549
Name: count, dtype: int64

In [17]:
definitions = pd.read_excel("../data/TG_Definitions.xlsx", sheet_name="Определения ТГ")

definitions.shape

(164, 8)

In [18]:
definitions.info()

<class 'pandas.DataFrame'>
RangeIndex: 164 entries, 0 to 163
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Парфимерия  52 non-null     str    
 1   Unnamed: 1  30 non-null     str    
 2   Unnamed: 2  81 non-null     str    
 3   Unnamed: 3  81 non-null     str    
 4   Unnamed: 4  0 non-null      float64
 5   Unnamed: 5  0 non-null      float64
 6   Unnamed: 6  130 non-null    str    
 7   Unnamed: 7  129 non-null    str    
dtypes: float64(2), str(6)
memory usage: 54.1 KB


In [19]:
definitions.head(10)

,Парфимерия,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,Указано в ППР № 1957,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Категория в КМТ,Продукция,ТН ВЭД,Наименование ТН ВЭД,NaN,NaN,ОКПД2,Наименование ОКПД2
2,Парфюмерия,"Духи, Туалетная вода, Одеколоны",3303 00,3303 Духи и туалетная вода,NaN,NaN,20.42.11,NaN
3,Косметика,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Указано в ППР № 1681,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Косметика для губ,Косметические средства или средства для макияж...,3304100000,3304100000 Средства для макияжа губ,NaN,NaN,20.42.12.110,20.42.12.110 Средства для макияжа губ
6,Косметика для глаз,NaN,3304200000,3304200000 Средства для макияжа глаз,NaN,NaN,20.42.12.120,20.42.12.120 Средства для макияжа глаз
7,Средства для маникюра и педикюра,NaN,3304300000,3304300000 Средства для маникюра или педикюра,NaN,NaN,20.42.13.000,20.42.13.000 Средства для маникюра или педикюра
8,Тональные средства и пудра,NaN,3304910000,"3304910000 Пудра, включая компактную",NaN,NaN,20.42.14.110,20.42.14.110 Пудры и крем-пудры
9,NaN,NaN,NaN,NaN,NaN,NaN,20.42.14.130,20.42.14.130 Тальк и прочие присыпки для детей


In [20]:
dictionary = pd.read_excel(
    "../data/TG_Definitions.xlsx",
    sheet_name="Справочник значений"
)

print(dictionary.shape)
dictionary.info()
dictionary.head(10)

(1315, 4)
<class 'pandas.DataFrame'>
RangeIndex: 1315 entries, 0 to 1314
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Наименование атрибута  1241 non-null   str   
 1   Значение атрибута      1238 non-null   object
 2   Категория(-и)          1004 non-null   str   
 3   ТНВЭД                  1062 non-null   str   
dtypes: object(1), str(3)
memory usage: 171.5+ KB


,Наименование атрибута,Значение атрибута,Категория(-и),ТНВЭД
0,Парфимерия,NaN,NaN,NaN
1,Тип парфюмерии,ДУХИ,NaN,NaN
2,Тип парфюмерии,ДУШИСТАЯ ВОДА,NaN,NaN
3,Тип парфюмерии,ЛАВАНДОВАЯ ВОДА,NaN,NaN
4,Тип парфюмерии,ОДЕКОЛОН,NaN,NaN
5,Тип парфюмерии,ПАРФЮМЕРНАЯ ВОДА,NaN,NaN
6,Тип парфюмерии,СПРЕЙ ДЛЯ ТЕЛА,NaN,NaN
7,Тип парфюмерии,ТУАЛЕТНАЯ ВОДА,NaN,NaN
8,Тип парфюмерии,КОМПЛЕКТ,NaN,NaN
9,NaN,NaN,NaN,NaN


In [21]:
tnved = pd.read_excel(
    "../data/TG_Definitions.xlsx",
    sheet_name="Коды ТНВЭД по категориям"
)

print(tnved.shape)
tnved.info()
tnved.head(10)

(76, 2)
<class 'pandas.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Парфимерия  76 non-null     object
 1   Unnamed: 1  74 non-null     str   
dtypes: object(1), str(1)
memory usage: 12.5+ KB


,Парфимерия,Unnamed: 1
0,Код ТНВЭД,Наименование кода ТНВЭД
1,3303001000,3303001000 Духи
2,3303009000,3303009000 Туалетная вода
3,Косметика,NaN
4,3304100000,3304100000 Средства для макияжа губ
5,3304200000,3304200000 Средства для макияжа глаз
6,3304300000,3304300000 Средства для маникюра или педикюра
7,3304910000,"3304910000 Пудра, включая компактную"
8,3304990000,3304990000 Прочие косметические средства или с...
9,3305100000,3305100000 Шампуни


In [22]:
okpd = pd.read_excel(
    "../data/TG_Definitions.xlsx",
    sheet_name="Коды ОКПД2 по категориям"
)

print(okpd.shape)
okpd.info()
okpd.head(10)

(62, 2)
<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Косметика   62 non-null     str  
 1   Unnamed: 1  62 non-null     str  
dtypes: str(2)
memory usage: 6.8 KB


,Косметика,Unnamed: 1
0,Код ОКПД2,Наименование кода ОКПД2
1,20.41.31.111,"20.41.31.111 Мыло туалетное марки ""Нейтральное"""
2,20.41.31.112,"20.41.31.112 Мыло туалетное марки ""Экстра"""
3,20.41.31.113,"20.41.31.113 Мыло туалетное марки ""Детское"""
4,20.41.31.114,"20.41.31.114 Мыло туалетное марки ""Ординарное"""
5,20.41.31.119,20.41.31.119 Мыло туалетное твердое прочее
6,20.41.31.121,20.41.31.121 Мыло хозяйственное I группы
7,20.41.31.122,20.41.31.122 Мыло хозяйственное II группы
8,20.41.31.123,20.41.31.123 Мыло хозяйственное III группы
9,20.41.31.130,20.41.31.130 Мыло туалетное жидкое


In [23]:
sample["number"] == case_df.loc[2, "rd_documentnumber"]

True

In [24]:
doc = json.loads(case_df.loc[2, "rd_data"])

doc["nameProd"]

'Биологически активная добавка к пище "КурКумин с витаминами D3 и B12" ("CoreCumin + Vitamins D3 & B12"), жидкость во флаконах по 5-100 мл с дозатором-капельницей или дозатором пипеткой. '

In [25]:
for i in range(5):
    doc = json.loads(case_df.loc[i, "rd_data"])
    print("-" * 50)
    print(doc.get("nameProd"))

--------------------------------------------------
None
--------------------------------------------------
None
--------------------------------------------------
Биологически активная добавка к пище "КурКумин с витаминами D3 и B12" ("CoreCumin + Vitamins D3 & B12"), жидкость во флаконах по 5-100 мл с дозатором-капельницей или дозатором пипеткой. 
--------------------------------------------------
Биологически активная добавка к пище «БИТ ИТ Спорт супер концентрат» («BEET IT Sport super concentrate»), жидкость в бутылках по 70-250 мл. 
--------------------------------------------------
Биологически активная добавка к пище «Глутамин Зеро» («GLUTAMINE  ZERO») со вкусами: или «Лимон», или «Голубой виноград», или «Персиковый чай со льдом», или «Арбуз».


In [26]:
sample_size = 10000

records = [
    json.loads(x)
    for x in case_df["rd_data"].head(sample_size)
]

json_df = pd.DataFrame(records)

In [27]:
json_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   number             10000 non-null  str    
 1   product            4469 non-null   object 
 2   type               9998 non-null   float64
 3   active             9996 non-null   object 
 4   docNorm            7087 non-null   str    
 5   useArea            9392 non-null   str    
 6   nameProd           9997 non-null   str    
 7   protocol           9996 non-null   str    
 8   firmGetName        9996 non-null   str    
 9   statusGroup        9996 non-null   float64
 10  firmMadeName       9997 non-null   str    
 11  activeFromDate     9996 non-null   str    
 12  dateFrom           4467 non-null   str    
 13  idStatus           4467 non-null   str    
 14  applicant          4467 non-null   object 
 15  RD_fullname        4469 non-null   str    
 16  certificate        4467 non-null  

In [28]:
json_df.isna().mean().sort_values()

number               0.0000
type                 0.0002
nameProd             0.0003
firmMadeName         0.0003
protocol             0.0004
statusGroup          0.0004
firmGetName          0.0004
active               0.0004
activeFromDate       0.0004
useArea              0.0608
docNorm              0.2913
product              0.5531
RD_fullname          0.5531
dateFrom             0.5533
idStatus             0.5533
applicant            0.5533
certificate          0.5533
declaration          0.5533
rpnStatusId          0.5533
manufacturer         0.5533
techRegulations      0.5533
certificationBody    0.5533
dtype: float64

In [29]:
json_df['type'].value_counts()

type
13.0    9998
Name: count, dtype: int64

In [30]:
json_df["statusGroup"].value_counts()

statusGroup
1.0    9581
2.0     415
Name: count, dtype: int64

In [31]:
json_df["nameProd"].str.len().describe()

count    9997.000000
mean      119.515355
std        75.724639
min        11.000000
25%        64.000000
50%       106.000000
75%       158.000000
max      2507.000000
Name: nameProd, dtype: float64

In [32]:
json_df["docNorm"].str.len().describe()

count    7087.000000
mean       33.659659
std        32.361813
min         0.000000
25%        15.000000
50%        29.000000
75%        44.000000
max       383.000000
Name: docNorm, dtype: float64

In [33]:
json_df["nameProd"].sample(10, random_state=42)

6252                         Ароматизатор «285744 Малины»
4684    Комплексная пищевая добавка КОМБИ СЕРВЕЛАТ ТРА...
1731    Специализированный пищевой продукт диетическог...
4742    Комплексная пищевая добавка МОЛОЧНАЯ ЭКСТРА, а...
4521    Краска стирол-акриловая водно-дисперсионная ВД...
6340             Ароматизатор пищевой «Клубника ELL 1864»
576     Биологически активная добавка к пище: «Комплек...
5202    Средство для для облегчения глажения изделий и...
6363             Натуральный ароматизатор «912166 Кетчуп»
439     Биологически активная добавка к пище «Железо х...
Name: nameProd, dtype: str

In [34]:
json_df["docNorm"].sample(10, random_state=42)

6252                                                  NaN
4684                                                     
1731                          спецификация производителя.
4742                                                     
4521    ТУ BY 200026232.001-2010, РЦ РБ 200026232.001-...
6340                              ТУ 9145-001-66969518-11
576                         ТУ 10.89.19-012-28920794-2024
5202    ТУ BY 200574861.018-2010, РЦ BY 200574861.104-...
6363                                                     
439                         ТУ 10.89.19-125-16013430-2021
Name: docNorm, dtype: str

In [35]:
json_df["useArea"].sample(10, random_state=42)

6252    при производстве чайного листа (пакетированног...
4684    производство пищевых продуктов (мясные, в т.ч....
1731    в качестве специализированного пищевого продук...
4742    производство пищевых продуктов (мясные, в т.ч....
4521                                                  NaN
6340                   при производстве пищевых продуктов
576     Условия реализации: места реализации определяю...
5202                                                     
6363    в пищевой промышленности в дозировке 4% при пр...
439     качестве БАД к пище - дополнительного источник...
Name: useArea, dtype: str

In [36]:
idx1 = json_df["product"].first_valid_index()
idx1

0

In [37]:
sample = json.loads(case_df.loc[idx1, "rd_data"])
type(sample["product"])

dict

In [38]:
pprint(sample["product"])

{'identification': [], 'productName': ''}


In [39]:
idx2 = json_df["manufacturer"].first_valid_index()
idx2

3

In [40]:
sample2 = json.loads(case_df.loc[idx2, "rd_data"])
pprint(sample2["manufacturer"])

{'GLN': '',
 'address': '',
 'filialAddresses': '',
 'inn': '',
 'name': '',
 'type': ''}


In [41]:
idx3 = json_df["certificate"].first_valid_index()
idx3

3

In [42]:
sample3 = json.loads(case_df.loc[idx3, "rd_data"])
pprint(sample3["certificate"])

{'certEndDate': '',
 'certRegDate': '',
 'filialAddresses': '',
 'idCertScheme': '',
 'idCertType': '',
 'issueBasis': '',
 'number': ''}


In [43]:
idx4 = json_df["declaration"].first_valid_index()
idx4
sample4 = json.loads(case_df.loc[idx4, "rd_data"])
pprint(sample4["declaration"])

{'declEndDate': '',
 'declRegDate': '',
 'filialAddresses': '',
 'idDeclScheme': '',
 'idDeclType': '',
 'number': ''}


In [44]:
sample["product"].keys()

dict_keys(['productName', 'identification'])

In [45]:
case_df["rd_documentnumber"].duplicated().sum()

np.int64(7714)

In [46]:
duplicates = (
    case_df
    .groupby("rd_documentnumber")["rank"]
    .agg(["count", "min", "max"])
)

duplicates.sort_values("count", ascending=False).head(20)

,count,min,max
rd_documentnumber,,,
ЕАЭС N RU Д-RU.РА01.В.23496/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.26974/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.35506/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.02855/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.23568/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.24237/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.23341/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.28429/21,4,1,4
ЕАЭС N RU Д-RU.РА01.В.37034/21,4,1,4


In [47]:
truth_df["rd_number"].sample(20, random_state=42).tolist()

['ЕАЭС N RU Д-KR.РА01.В.87925/23',
 'ЕАЭС N RU Д-RU.РА02.В.14528/23',
 'ЕАЭС N RU Д-RU.РА07.В.80755/22',
 'ЕАЭС N RU Д-RU.РА02.В.95459/21',
 'ЕАЭС N RU Д-RU.РА09.В.53146/24',
 'ЕАЭС N RU Д-RU.РА04.В.11412/25',
 'ЕАЭС N RU Д-DE.РА06.В.27005/23',
 'ЕАЭС N RU Д-RU.РА10.В.77082/23',
 'ЕАЭС N RU Д-RU.РА05.В.26694/24',
 'RU.30.АЦ.02.015.Е.000170.04.24',
 'RU.54.НС.01.015.Е.000985.11.23',
 'ЕАЭС N RU Д-RU.РА02.В.45118/23',
 'ЕАЭС N RU Д-RU.РА01.В.41529/25',
 'ЕАЭС N RU Д-RU.РА07.В.39083/23',
 'ЕАЭС № BY/112 11.01. ТР009 011 06175',
 'ЕАЭС N RU Д-RU.РА02.В.83266/24',
 'ЕАЭС N RU Д-BE.РА10.В.58348/23 ||| ЕАЭС N RU Д-BE.РА11.В.25898/25',
 'ЕАЭС N RU Д-DE.РА01.В.60876/25',
 'KG.11.01.09.001.R.005950.11.24',
 'ЕАЭС N RU Д-RU.РА07.В.16997/25']

In [48]:
import random

sample_numbers = case_df["rd_documentnumber"].dropna().sample(10, random_state=42)

sample_numbers.tolist()

['ЕАЭС N RU Д-RU.РА04.В.67842/23',
 'ЕАЭС AM-020/S.B-0044-2024',
 'ЕАЭС N RU Д-PL.СП28.В.10672',
 'ЕАЭС № BY/112 11.02. ТР017 122 00265',
 'RU.77.99.32.009.Е.003170.02.15',
 'ТС RU С-TR.АУ04.В.04607',
 'ЕАЭС N RU Д-RU.РА06.А.44458/25',
 'ЕАЭС N RU Д-RU.АЖ24.В.00857',
 'ЕАЭС RU С-TJ.НЕ16.В.00405/21',
 'ЕАЭС N RU Д-RU.РА10.В.20651/25']

### Историческая проверка

На первоначальной версии Truth прямое сопоставление не работало.
После исправления `rd_number` экспертом данный эксперимент больше
не используется в основном анализе.

In [49]:
for number in sample_numbers:
    matches = truth_df["rd_number"].str.contains(
        number,
        regex=False,
        na=False
    ).sum()
    
    print(number, '->', matches)

ЕАЭС N RU Д-RU.РА04.В.67842/23 -> 0
ЕАЭС AM-020/S.B-0044-2024 -> 0
ЕАЭС N RU Д-PL.СП28.В.10672 -> 0
ЕАЭС № BY/112 11.02. ТР017 122 00265 -> 0
RU.77.99.32.009.Е.003170.02.15 -> 0
ТС RU С-TR.АУ04.В.04607 -> 0
ЕАЭС N RU Д-RU.РА06.А.44458/25 -> 0
ЕАЭС N RU Д-RU.АЖ24.В.00857 -> 0
ЕАЭС RU С-TJ.НЕ16.В.00405/21 -> 0
ЕАЭС N RU Д-RU.РА10.В.20651/25 -> 0


In [50]:
for i, row in enumerate(case_df["rd_data"].head(50000)):
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        ident = product.get("identification")

        if ident:
            print(i)
            print(ident)
            break

In [51]:
from collections import Counter
import json

counter = Counter()

for row in case_df["rd_data"].head(10000):
    data = json.loads(row)
    counter[tuple(sorted(data.keys()))] += 1

counter.most_common(10)

[(('RD_fullname',
   'active',
   'activeFromDate',
   'applicant',
   'certificate',
   'certificationBody',
   'dateFrom',
   'declaration',
   'docNorm',
   'firmGetName',
   'firmMadeName',
   'idStatus',
   'manufacturer',
   'nameProd',
   'number',
   'product',
   'protocol',
   'rpnStatusId',
   'statusGroup',
   'techRegulations',
   'type',
   'useArea'),
  4467),
 (('active',
   'activeFromDate',
   'firmGetName',
   'firmMadeName',
   'nameProd',
   'number',
   'protocol',
   'statusGroup',
   'type',
   'useArea'),
  2642),
 (('active',
   'activeFromDate',
   'docNorm',
   'firmGetName',
   'firmMadeName',
   'nameProd',
   'number',
   'protocol',
   'statusGroup',
   'type',
   'useArea'),
  2282),
 (('active',
   'activeFromDate',
   'docNorm',
   'firmGetName',
   'firmMadeName',
   'nameProd',
   'number',
   'protocol',
   'statusGroup',
   'type'),
  337),
 (('active',
   'activeFromDate',
   'firmGetName',
   'firmMadeName',
   'nameProd',
   'number',
   'proto

In [52]:
json_df["product"].dropna().head(10)

0             {'productName': '', 'identification': []}
1             {'productName': '', 'identification': []}
3     {'tnved': '', 'productInfo': '', 'productName'...
4     {'tnved': '', 'productInfo': '', 'productName'...
5     {'tnved': '', 'productInfo': '', 'productName'...
6     {'tnved': '', 'productInfo': '', 'productName'...
7     {'tnved': '', 'productInfo': '', 'productName'...
8     {'tnved': '', 'productInfo': '', 'productName'...
9     {'tnved': '', 'productInfo': '', 'productName'...
10    {'tnved': '', 'productInfo': '', 'productName'...
Name: product, dtype: object

In [53]:
from collections import Counter
import json

product_keys = Counter()

for row in case_df["rd_data"].head(10000):
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        product_keys[tuple(sorted(product.keys()))] += 1

product_keys.most_common(10)

[(('idObjectType',
   'idProductOrigin',
   'identification',
   'productInfo',
   'productName',
   'tnved'),
  4467),
 (('identification', 'productName'), 2)]

In [54]:
import json

for row in case_df["rd_data"]:
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        print(product.keys())
        break

dict_keys(['productName', 'identification'])


In [55]:
from collections import Counter

object_types = Counter()

for row in case_df["rd_data"]:
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        object_types[product.get("idObjectType")] += 1

object_types.most_common()

[('Серийный выпуск', 2232563),
 ('Партия', 594064),
 ('Единичное изделие', 437529),
 ('', 121295),
 (None, 139)]

In [56]:
object_types = Counter()

for row in case_df["rd_data"]:
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        object_types[product.get("idProductOrigin")] += 1

object_types.most_common()

[('РОССИЯ', 1170029),
 ('', 937743),
 (None, 309802),
 ('КИТАЙ', 252255),
 ('ИТАЛИЯ', 95426),
 ('ГЕРМАНИЯ', 86228),
 ('ФРАНЦИЯ', 63954),
 ('БЕЛАРУСЬ', 63344),
 ('ТУРЦИЯ', 45099),
 ('КОРЕЯ, РЕСПУБЛИКА', 36560),
 ('СОЕДИНЕННЫЕ ШТАТЫ', 30539),
 ('ЯПОНИЯ', 22116),
 ('ИСПАНИЯ', 21828),
 ('СОЕДИНЕННОЕ КОРОЛЕВСТВО', 20414),
 ('ПОЛЬША', 17086),
 ('УЗБЕКИСТАН', 14538),
 ('НИДЕРЛАНДЫ', 12466),
 ('ШВЕЙЦАРИЯ', 12219),
 ('ИНДИЯ', 11562),
 ('ВЬЕТНАМ', 9260),
 ('ГОНКОНГ', 8474),
 ('ТАЙВАНЬ (КИТАЙ)', 8395),
 ('ФИНЛЯНДИЯ', 8057),
 ('БЕЛЬГИЯ', 7584),
 ('ГРЕЦИЯ', 7188),
 ('ИРАН, ИСЛАМСКАЯ РЕСПУБЛИКА', 7068),
 ('УКРАИНА', 5630),
 ('ШВЕЦИЯ', 5533),
 ('ЧЕХИЯ', 5041),
 ('ТАИЛАНД', 4428),
 ('АВСТРИЯ', 4194),
 ('ИЗРАИЛЬ', 4158),
 ('ДАНИЯ', 3787),
 ('СЛОВЕНИЯ', 3758),
 ('БАНГЛАДЕШ', 3735),
 ('КИРГИЗИЯ', 3682),
 ('КАЗАХСТАН', 3449),
 ('КАНАДА', 3362),
 ('ИНДОНЕЗИЯ', 2880),
 ('ИРЛАНДИЯ', 2756),
 ('СЕРБИЯ', 2496),
 ('БОЛГАРИЯ', 2479),
 ('ОБЪЕДИНЕННЫЕ АРАБСКИЕ ЭМИРАТЫ', 2232),
 ('МАЛАЙЗИЯ', 2146),
 ('МЕКСИКА', 1909

In [57]:
import json

best = None
best_len = 0

for row in case_df["rd_data"]:
    data = json.loads(row)

    product = data.get("product")

    if isinstance(product, dict):
        if len(product) > best_len:
            best_len = len(product)
            best = product

print(best_len)
print(best.keys())

for k, v in best.items():
    print("=" * 40)
    print(k)
    print(type(v))
    print(v)

6
dict_keys(['tnved', 'productInfo', 'productName', 'idObjectType', 'identification', 'idProductOrigin'])
tnved
<class 'str'>

productInfo
<class 'str'>

productName
<class 'str'>

idObjectType
<class 'str'>

identification
<class 'list'>
[]
idProductOrigin
<class 'str'>



In [58]:
import json

max_keys = 0
best = None

for row in case_df["rd_data"]:
    data = json.loads(row)

    for key, value in data.items():
        if isinstance(value, dict):
            if len(value) > max_keys:
                max_keys = len(value)
                best = value

print(max_keys)
print(best.keys())

8
dict_keys(['inn', 'ogrn', 'type', 'email', 'phone', 'address', 'fullName', 'directorName'])


In [59]:
perfume_keywords = ["духи", 
                    "парфюмерная вода", 
                    "туалетная вода", 
                    "одеколон",
                    "parfum",
                    "perfume"]

cosmetic_keywords = [
    "крем",
    "шампун",
    "лосьон",
    "бальзам",
    "маска"
]

oil_keywords = [
    "моторное масло",
    "масло моторное",
    "5w",
    "10w",
    "sae",
    "api"
]

In [60]:
def count_keyword(text_series, keywords):
    text = text_series.fillna("").str.lower()
    
    result = {}

    for word in keywords:
        count = text.str.contains(word, regex=False).sum()
        result[word] = count
        
    return (
        pd.Series(result)
        .sort_values(ascending=False)
    )

## Гипотеза: простые ключевые слова в nameProd

Гипотеза не подтверждена в исходном виде.

Причины:
- выборка;
- низкое покрытие TG4;
- ложные совпадения (`крем` и т.п.);
- необходимость учитывать контекст и комбинации.

Вывод:
простое наличие одиночного слова не считаем достаточной feature.

In [61]:
print("===Парфюмерия===")
display(count_keyword(json_df["nameProd"], perfume_keywords))

print("===Косметика===")
display(count_keyword(json_df["nameProd"], cosmetic_keywords))

print("===Моторные масла===")
display(count_keyword(json_df["nameProd"], oil_keywords))

===Парфюмерия===


parfum              2
туалетная вода      1
парфюмерная вода    0
духи                0
одеколон            0
perfume             0
dtype: int64

===Косметика===


крем       384
шампун     111
бальзам     81
лосьон      79
маска       39
dtype: int64

===Моторные масла===


api               11
5w                 1
масло моторное     0
моторное масло     0
10w                0
sae                0
dtype: int64

In [62]:
json_df.loc[
    json_df["nameProd"]
        .fillna("")
        .str.lower()
        .str.contains("духи"),
    "nameProd"
].head(10)

Series([], Name: nameProd, dtype: str)

In [63]:
sample_size = 10000

sample_case = (
    case_df
    .sample(sample_size, random_state=42)
    .reset_index(drop=True)
)

records = [
    json.loads(x)
    for x in sample_case["rd_data"]
]

json_df = pd.DataFrame(records)

In [64]:
print("=== Парфюмерия ===")
display(count_keyword(json_df["nameProd"], perfume_keywords))

print("=== Косметика ===")
display(count_keyword(json_df["nameProd"], cosmetic_keywords))

print("=== Моторные масла ===")
display(count_keyword(json_df["nameProd"], oil_keywords))

=== Парфюмерия ===


perfume             3
parfum              2
одеколон            1
духи                0
парфюмерная вода    0
туалетная вода      0
dtype: int64

=== Косметика ===


крем       89
шампун     25
лосьон     21
маска      18
бальзам    13
dtype: int64

=== Моторные масла ===


api               2
моторное масло    0
масло моторное    0
5w                0
10w               0
sae               0
dtype: int64

In [65]:
for tg in [4, 35, 43]:
    print("=" * 80)
    print("TG =", tg)
    print()

    display(
        truth_df.loc[
            truth_df["tg"] == str(tg),
            "rd_number"
        ].head(10)
    )

TG = 4



0    Декларация о соответствии распространяется на ...
1    ГОСТ 31678-2012 Продукция парфюмерная жидкая. ...
2    Условия хранения стандартные для данного вида ...
3    Дата изготовления отобранных образцов (проб) п...
4    ГОСТ 32893-2014 Продукция парфюмерно-косметиче...
5    "ГОСТ 31678-2012 ""Продукция парфюмерная жидка...
6    Декларация соответствия распространяется на пр...
7    Green Almond & Redcurrant, торговой марки Jo M...
8      - Парфюмерная вода для мужчин DAVID BECKHAM ...
9    Декларация соответствия распространяется на пр...
Name: rd_number, dtype: str

TG = 35



224189    ЕАЭС N RU Д-RU.РА02.В.80355/22
224190    RU.77.01.34.001.Е.002952.12.16
224191    KG.11.01.09.001.R.006012.10.22
224192    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224193    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224194    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224195    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224196    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224197    ЕАЭС N RU Д-RU.ПК08.В.01644/20
224198    ЕАЭС N RU Д-RU.ПК08.В.01644/20
Name: rd_number, dtype: str

TG = 43



549    ЕАЭС N RU Д-FR.РА04.В.75724/22
550    ЕАЭС N RU Д-RU.РА09.В.31300/22
551    ЕАЭС N RU Д-RU.РА05.В.60070/22
552    ЕАЭС N RU Д-RU.РА05.В.10364/23
553    ЕАЭС N RU Д-RU.РА05.В.10364/23
554    ЕАЭС N RU Д-RU.РА05.В.10364/23
555    ЕАЭС N RU Д-RU.РА05.В.10364/23
556    ЕАЭС N RU Д-RU.РА05.В.10364/23
557    ЕАЭС N RU Д-RU.РА05.В.10364/23
558    ЕАЭС N RU Д-RU.РА05.В.10364/23
Name: rd_number, dtype: str

In [66]:
truth_df.groupby("tg")["rd_type"].value_counts()

tg  rd_type
35  ДС         259223
    СГР         86947
    СС           2995
4   N/A           549
43  ДС         221884
    СГР          1026
    СС            730
Name: count, dtype: int64

In [67]:
truth_df.loc[
    truth_df["tg"] == "4",
    "rd_number"
].sample(20, random_state=42)

195    Декларация о соответствии распространяется на ...
79      - Парфюмерная вода для женщин Calvin Klein Et...
479    "п.п. 3.1.1, 3.1.5, 3.1.6; 3.2; 3.3.1; 3.4.3, ...
109    Продукция парфюмерно-косметическая жидкая: оде...
473                                          ""FEDERAL""
490    Декларация о соответствии распространяется на ...
84     Договор поставки № KGD-1728 от 27.01.2023 года...
368    Условия хранения стандартные для данного вида ...
132    Декларация о соответствии распространяется на ...
364                                           20.06.2023
184    Декларация соответствия распространяется на пр...
10     Kirke, торговой марки Tiziana Terenzi  Cruz de...
73     Декларация о соответствии распространяется на ...
220                                              SAFANAD
278    "Декларация о соответствии распространяется на...
82     "ГОСТ 31678-2012 ""Продукция парфюмерная жидка...
382    Декларация соответствия распространяется на пр...
6      Декларация соответствия 

In [68]:
truth_df.groupby("tg")["code_tnved"].nunique()

tg
35    74
4      2
43    32
Name: code_tnved, dtype: int64

In [69]:
for tg in ["4", "35", "43"]:
    print("=" * 80)
    print("TG =", tg)

    print(
        truth_df.loc[
            truth_df["tg"] == tg,
            "code_tnved"
        ]
        .value_counts()
        .head(15)
    )

TG = 4
code_tnved
3303009000    335
3303001000    161
Name: count, dtype: int64
TG = 35
code_tnved
3304990000    82940
3305900009    47024
3402500000    43973
3304300000    35844
3401300000    26808
3305100000    18077
3307490000    16091
3304100000    11343
3307300000    11002
3401110001    10350
3304200000     7677
3401209000     7271
3808948000     5497
3307200000     4496
3307900008     2907
Name: count, dtype: int64
TG = 43
code_tnved
2710198200           97609
2710198800           37186
3403199000           32767
3820000000           25145
3403990000           19389
3403191000            7653
3819000000            3808
2710                    18
 VW 502.00/505.00       11
 VW 501.00/505.00        8
3403                     7
 MB 229.51/"""           7
 Renault 0700            5
 ACEA A3                 3
 PO"""                   3
Name: count, dtype: int64


# Получили обновлённый датасет truth

In [70]:
new_truth = pd.read_parquet("../data/2_truth_rd_data.snappy.parquet")

In [71]:
new_truth

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
0,4,N/A,3303,3303009000,ЕАЭС N RU Д-IT.РА03.В.08011/25,2025-03-25
1,4,N/A,3303,3303009000,ЕАЭС N RU Д-FR.РА01.В.63599/25,2025-02-03
2,4,N/A,3303,3303009000,ЕАЭС N RU Д-FR.РА08.В.83499/24,2024-09-27
3,4,N/A,3303,3303009000,ЕАЭС N RU Д-ES.РА02.В.52976/25,2025-02-28
4,4,N/A,3303,3303009000,ЕАЭС N RU Д-ES.РА09.В.91241/23,2023-11-22
...,...,...,...,...,...,...
577255,35,СГР,3307,3307490000,RU.08.08.09.015.Е.000383.10.25,NaN
577256,35,СГР,3307,3307490000,RU.08.08.09.015.Е.000383.10.25,NaN
577257,35,СГР,3307,3307490000,RU.08.08.09.015.Е.000383.10.25,NaN
577258,35,СГР,3307,3307490000,RU.08.08.09.015.Е.000383.10.25,NaN


In [72]:
sample_truth = new_truth["rd_number"].sample(100, random_state=42)

matches = []

for number in sample_truth:
    if (case_df["rd_documentnumber"] == number).any():
        matches.append(number)
        
print(len(matches))        

94


In [73]:
new_truth["rd_date"].isna().mean()

np.float64(0.6066936908845234)

In [74]:
new_truth.groupby("tg")["rd_date"].apply(lambda x: x.notna().mean())

tg
35    0.000000
4     0.999776
43    0.995287
Name: rd_date, dtype: float64

In [75]:
matches = []

case_lookup = (
    case_df
    .set_index("rd_documentnumber")
)

for number in sample_truth:
    if number in case_lookup.index:
        matches.append(number)

matches[:5]

['ЕАЭС N RU Д-RU.РА01.В.52411/21',
 'ВП RU Д-AE.РА01.А.70053/25',
 'ЕАЭС N RU Д-CN.РА03.В.78441/25',
 'ЕАЭС N RU Д-KR.РА01.В.89075/23',
 'RU.30.АЦ.02.015.Е.000307.07.25']

In [76]:
number = matches[0]

case_df.loc[
    case_df["rd_documentnumber"] == number,
    ["rd_documentnumber", "rank"]
]

,rd_documentnumber,rank
2043160,ЕАЭС N RU Д-RU.РА01.В.52411/21,1
2043161,ЕАЭС N RU Д-RU.РА01.В.52411/21,2


In [77]:
new_truth.loc[
    new_truth["rd_number"] == number
]

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
351494,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351495,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351496,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351526,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351527,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN
351528,35,ДС,3401,3401300000,ЕАЭС N RU Д-RU.РА01.В.52411/21,NaN


In [78]:
case_df.head(10)

,rd_documentnumber,rank,rd_data,tg_ids
0,,1,"{""number"": """", ""product"": {""productName"": """", ...",[]
1,,2,"{""number"": """", ""product"": {""productName"": """", ...",[]
2,AM.01.01.01.003.R.000013.01.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[35]
3,AM.01.01.01.003.R.000014.01.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
4,AM.01.01.01.003.R.000018.07.20,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
5,AM.01.01.01.003.R.000021.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
6,AM.01.01.01.003.R.000022.02.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[37]
7,AM.01.01.01.003.R.000023.02.22,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[45]
8,AM.01.01.01.003.R.000025.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....","[45, 37]"
9,AM.01.01.01.003.R.000029.01.23,1,"{""type"": 13, ""active"": true, ""number"": ""AM.01....",[45]


In [79]:
case_rank1 = case_df[case_df["rank"] == 1]

print("Всего строк case_df:", len(case_df))
print("Строк rank == 1:", len(case_rank1))
print("Уникальных документов rank == 1:",
      case_rank1["rd_documentnumber"].nunique())


Всего строк case_df: 3603517
Строк rank == 1: 3595803
Уникальных документов rank == 1: 3595803


In [80]:
dup_rank1 = (
    case_rank1["rd_documentnumber"]
    .duplicated()
    .sum()
)

dup_rank1

np.int64(0)

In [81]:
case_rank1["rd_documentnumber"].value_counts().head(20)

rd_documentnumber
                                  1
AM.01.01.01.003.R.000013.01.22    1
AM.01.01.01.003.R.000014.01.22    1
AM.01.01.01.003.R.000018.07.20    1
AM.01.01.01.003.R.000021.01.23    1
AM.01.01.01.003.R.000022.02.22    1
AM.01.01.01.003.R.000023.02.22    1
AM.01.01.01.003.R.000025.01.23    1
AM.01.01.01.003.R.000029.01.23    1
AM.01.01.01.003.R.000030.02.22    1
AM.01.01.01.003.R.000031.02.22    1
AM.01.01.01.003.R.000034.01.23    1
AM.01.01.01.003.R.000035.02.22    1
AM.01.01.01.003.R.000040.02.22    1
AM.01.01.01.003.R.000041.01.24    1
AM.01.01.01.003.R.000042.02.22    1
AM.01.01.01.003.R.000045.01.23    1
AM.01.01.01.003.R.000047.01.23    1
AM.01.01.01.003.R.000054.02.23    1
AM.01.01.01.003.R.000056.12.20    1
Name: count, dtype: int64

In [82]:
print("Всего Truth rows:", len(new_truth))
print("Уникальных rd_number:", new_truth["rd_number"].nunique())

Всего Truth rows: 577260
Уникальных rd_number: 97822


In [83]:
truth_dup_counts = (
    new_truth["rd_number"]
    .value_counts()
)

truth_dup_counts.head(20)

rd_number
ЕАЭС N RU Д-DE.РА08.В.64999/22    4203
ЕАЭС N RU Д-RU.РА07.В.39083/23    2421
ЕАЭС N RU Д-DE.РА08.В.64358/22    2113
ЕАЭС N RU Д-RU.РА01.В.19119/24    1994
ЕАЭС N RU Д-DE.РА03.В.43293/25    1947
ЕАЭС N RU Д-RU.РА07.В.91691/25    1668
ЕАЭС N RU Д-RU.РА07.В.92846/25    1280
ЕАЭС N RU Д-JP.РА06.В.36735/23    1203
ЕАЭС N RU Д-RU.РА08.В.64705/22    1048
ЕАЭС N RU Д-RU.РА05.В.32856/25    1038
ЕАЭС N RU Д-RU.РА04.В.53695/24     997
ЕАЭС N RU Д-RU.РА03.В.26916/22     984
ЕАЭС N RU Д-KR.РА07.В.59450/23     966
ЕАЭС N RU Д-RU.РА06.В.22172/23     959
ЕАЭС N RU Д-RU.РА02.В.17177/22     877
ЕАЭС N RU Д-RU.РА08.В.61808/22     876
ЕАЭС N RU Д-RU.РА02.В.57852/24     858
ЕАЭС N RU Д-RU.РА04.В.64383/24     822
ЕАЭС N RU Д-RU.РА03.В.57296/24     815
ЕАЭС N RU Д-RU.РА10.В.62567/24     724
Name: count, dtype: int64

In [84]:
(truth_dup_counts > 1).sum()

np.int64(54349)

In [85]:
duplicated_numbers = truth_dup_counts[truth_dup_counts > 1].head(10).index

new_truth.loc[
    new_truth["rd_number"].isin(duplicated_numbers)
].sort_values("rd_number")

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
4598,43,ДС,2710,2710198200,ЕАЭС N RU Д-DE.РА03.В.43293/25,2025-04-07
4599,43,ДС,2710,2710198200,ЕАЭС N RU Д-DE.РА03.В.43293/25,2025-04-07
4603,43,ДС,2710,2710198200,ЕАЭС N RU Д-DE.РА03.В.43293/25,2025-04-07
4619,43,ДС,2710,2710198200,ЕАЭС N RU Д-DE.РА03.В.43293/25,2025-04-07
4632,43,ДС,2710,2710198200,ЕАЭС N RU Д-DE.РА03.В.43293/25,2025-04-07
...,...,...,...,...,...,...
477515,35,ДС,3304,3304300000,ЕАЭС N RU Д-RU.РА08.В.64705/22,NaN
477516,35,ДС,3304,3304300000,ЕАЭС N RU Д-RU.РА08.В.64705/22,NaN
477517,35,ДС,3304,3304300000,ЕАЭС N RU Д-RU.РА08.В.64705/22,NaN
477518,35,ДС,3304,3304300000,ЕАЭС N RU Д-RU.РА08.В.64705/22,NaN


In [86]:
truth_numbers = set(new_truth["rd_number"].dropna())
case_numbers_rank1 = set(
    case_rank1["rd_documentnumber"].dropna()
)

matched = truth_numbers & case_numbers_rank1

print("Уникальных Truth документов:", len(truth_numbers))
print("Найдено в RD rank=1:", len(matched))
print("Не найдено:", len(truth_numbers - case_numbers_rank1))

Уникальных Truth документов: 97822
Найдено в RD rank=1: 86301
Не найдено: 11521


In [87]:
len(matched) / len(truth_numbers)

0.8822248573940422

In [88]:
sample_numbers = list(matched)[:1000]

sample_case = case_rank1[
    case_rank1["rd_documentnumber"].isin(sample_numbers)
][["rd_documentnumber", "rd_data"]].copy()

In [89]:
import json

sample_case["activeFromDate"] = sample_case["rd_data"].map(
    lambda x: json.loads(x).get("activeFromDate")
)

In [90]:
sample_case[[
    "rd_documentnumber",
    "activeFromDate"
]].head()

,rd_documentnumber,activeFromDate
2170,AM.02.07.01.001.R.000137.02.24,2024-02-09
2400,AM.02.07.01.001.R.000384.04.22,2022-04-20
14335,BY.50.51.01.001.Е.000599.10.18,2018-10-11
21682,BY.70.06.01.001.R.001203.04.21,2021-04-28
22976,BY.70.06.01.001.Е.000112.01.18,2018-01-12


In [91]:
truth_dates = new_truth[
    new_truth["rd_number"].isin(sample_numbers)
][[
    "rd_number",
    "tg",
    "rd_date"
]]

In [92]:
check_dates = sample_case.merge(
    truth_dates,
    left_on="rd_documentnumber",
    right_on="rd_number",
    how="inner"
)

check_dates.head()

,rd_documentnumber,rd_data,activeFromDate,rd_number,tg,rd_date
0,AM.02.07.01.001.R.000137.02.24,"{""type"": 13, ""active"": true, ""number"": ""AM.02....",2024-02-09,AM.02.07.01.001.R.000137.02.24,35,NaN
1,AM.02.07.01.001.R.000137.02.24,"{""type"": 13, ""active"": true, ""number"": ""AM.02....",2024-02-09,AM.02.07.01.001.R.000137.02.24,35,NaN
2,AM.02.07.01.001.R.000137.02.24,"{""type"": 13, ""active"": true, ""number"": ""AM.02....",2024-02-09,AM.02.07.01.001.R.000137.02.24,35,NaN
3,AM.02.07.01.001.R.000137.02.24,"{""type"": 13, ""active"": true, ""number"": ""AM.02....",2024-02-09,AM.02.07.01.001.R.000137.02.24,35,NaN
4,AM.02.07.01.001.R.000137.02.24,"{""type"": 13, ""active"": true, ""number"": ""AM.02....",2024-02-09,AM.02.07.01.001.R.000137.02.24,35,NaN


In [93]:
check_dates["date_match"] = (
    check_dates["activeFromDate"].astype("string").str[:10]
    ==
    check_dates["rd_date"].astype("string").str[:10]
)

check_dates["date_match"].value_counts(dropna=False)

date_match
<NA>     3568
False    2560
Name: count, dtype: int64[pyarrow]

In [94]:
new_truth.duplicated().sum()

np.int64(472101)

In [95]:
new_truth.duplicated(
    subset=[
        "rd_number",
        "tg",
        "rd_type",
        "group_tnved",
        "code_tnved",
        "rd_date"
    ]
).sum()

np.int64(472101)

In [96]:
truth_tg_per_doc = (
    new_truth
    .groupby("rd_number")["tg"]
    .nunique()
)

truth_tg_per_doc.value_counts().sort_index()

tg
1    97746
2       76
Name: count, dtype: int64

In [97]:
truth_tg_combinations = (
    new_truth
    .groupby("rd_number")["tg"]
    .agg(lambda x: tuple(sorted(set(x))))
)

truth_tg_combinations.value_counts()

tg
(35,)       77341
(43,)       16496
(4,)         3909
(35, 43)       43
(35, 4)        32
(4, 43)         1
Name: count, dtype: int64

In [98]:
truth_unique = new_truth.drop_duplicates()

In [99]:
truth_rows_per_doc = (
    truth_unique
    .groupby("rd_number")
    .size()
)

truth_rows_per_doc.value_counts().sort_index().head(20)

1     92270
2      4331
3       854
4       254
5        71
6        29
7         9
12        2
13        1
20        1
Name: count, dtype: int64

In [100]:
multi_truth = truth_rows_per_doc[truth_rows_per_doc > 1].index

truth_unique.loc[
    truth_unique["rd_number"].isin(multi_truth)
].groupby("rd_number").agg(
    tg_nunique=("tg", "nunique"),
    type_nunique=("rd_type", "nunique"),
    group_tnved_nunique=("group_tnved", "nunique"),
    code_tnved_nunique=("code_tnved", "nunique"),
    date_nunique=("rd_date", "nunique"),
).head(30)

,tg_nunique,type_nunique,group_tnved_nunique,code_tnved_nunique,date_nunique
rd_number,,,,,
"""""Greenleaf""""""",1,1,2,3,0
"Almond - linea JOC COLOR. BAREX""",1,1,1,2,0
Гель для душа «Настоящий мужчина»,1,1,2,2,0
"Лаванда «Reva Care», «Green Drago», """"Lavandel""""""",1,1,2,2,0
"масло усьмы для роста ресниц и бровей, сыворотка для роста ресниц и бровей, гель для бровей фиксирующий",1,1,3,4,0
1,1,1,2,2,1
168854412,1,1,1,2,1
249060612,1,1,19,3,1
4,1,1,1,2,1


In [101]:
truth_unique.loc[
    truth_unique["rd_number"].isin(multi_truth)
].groupby("rd_number").agg(
    rows=("rd_number", "size"),
    tg_nunique=("tg", "nunique"),
    code_tnved_nunique=("code_tnved", "nunique"),
)

,rows,tg_nunique,code_tnved_nunique
rd_number,,,
"""""Greenleaf""""""",3,1,3
"Almond - linea JOC COLOR. BAREX""",2,1,2
Гель для душа «Настоящий мужчина»,2,1,2
"Лаванда «Reva Care», «Green Drago», """"Lavandel""""""",2,1,2
"масло усьмы для роста ресниц и бровей, сыворотка для роста ресниц и бровей, гель для бровей фиксирующий",4,1,4
...,...,...,...
ТС N RU Д-TR.АГ95.В.00823,2,1,2
ТС RU С-AB.ЛД04.В.01465,3,1,3
ТС RU С-AD.АЛ16.А.13164,2,1,2


In [102]:
multi_truth_summary = (
    truth_unique
    .groupby("rd_number")
    .agg(
        rows=("rd_number", "size"),
        tg_nunique=("tg", "nunique"),
        code_tnved_nunique=("code_tnved", "nunique"),
    )
)

print("Документов с несколькими строками:")
print((multi_truth_summary["rows"] > 1).sum())

print("\n1 TG + несколько TNVED:")
print((
    (multi_truth_summary["tg_nunique"] == 1) &
    (multi_truth_summary["code_tnved_nunique"] > 1)
).sum())

print("\nНесколько TG:")
print((multi_truth_summary["tg_nunique"] > 1).sum())

print("\nНесколько TG + несколько TNVED:")
print((
    (multi_truth_summary["tg_nunique"] > 1) &
    (multi_truth_summary["code_tnved_nunique"] > 1)
).sum())

Документов с несколькими строками:
5552

1 TG + несколько TNVED:
5203

Несколько TG:
76

Несколько TG + несколько TNVED:
73


In [103]:
pd.crosstab(
    multi_truth_summary["tg_nunique"],
    multi_truth_summary["code_tnved_nunique"]
)

code_tnved_nunique,0,1,2,3,4,5,6,7,12
tg_nunique,,,,,,,,,
1,115,92428,4027,820,246,71,29,8,2
2,0,3,57,12,3,0,1,0,0


In [104]:
same_tg_low_codes = multi_truth_summary[
    (multi_truth_summary["rows"] > 1) &
    (multi_truth_summary["tg_nunique"] == 1) &
    (multi_truth_summary["code_tnved_nunique"] <= 1)
].index

len(same_tg_low_codes)

273

In [105]:
suspicious = truth_unique.loc[
    truth_unique["rd_number"].isin(same_tg_low_codes)
].groupby("rd_number").agg(
    rows=("rd_number", "size"),
    tg_nunique=("tg", "nunique"),
    rd_type_nunique=("rd_type", "nunique"),
    group_tnved_nunique=("group_tnved", "nunique"),
    code_tnved_nunique=("code_tnved", "nunique"),
    rd_date_nunique=("rd_date", "nunique")
)

suspicious.head(30)

,rows,tg_nunique,rd_type_nunique,group_tnved_nunique,code_tnved_nunique,rd_date_nunique
rd_number,,,,,,
ЕАЭС KZ 7500361.13.12.15276,2,1,1,1,1,1
ЕАЭС N RU Д-AE.ПК08.В.01718/20,2,1,1,1,1,1
ЕАЭС N RU Д-AE.РА01.В.15416/21,2,1,1,1,1,1
ЕАЭС N RU Д-AE.РА01.В.43979/24,2,1,1,1,1,1
ЕАЭС N RU Д-AE.РА01.В.54968/24,2,1,1,1,1,1
ЕАЭС N RU Д-AE.РА02.В.77596/25,2,1,1,1,1,1
ЕАЭС N RU Д-AE.РА02.В.97772/24,2,1,1,1,1,1
ЕАЭС N RU Д-AE.РА04.В.75316/22,2,1,1,1,1,1
ЕАЭС N RU Д-AE.РА05.В.62389/23,2,1,1,1,1,1


In [106]:
example_numbers = same_tg_low_codes[:3]

truth_unique.loc[
    truth_unique["rd_number"].isin(example_numbers)
].sort_values("rd_number")

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
400,4,N/A,3303,3303009000,ЕАЭС KZ 7500361.13.12.15276,2023-04-13
4121,4,N/A,NaN,NaN,ЕАЭС KZ 7500361.13.12.15276,2023-04-13
1974,4,N/A,3303,3303009000,ЕАЭС N RU Д-AE.ПК08.В.01718/20,2020-11-03
4359,4,N/A,NaN,NaN,ЕАЭС N RU Д-AE.ПК08.В.01718/20,2020-11-03
104,4,N/A,3303,3303001000,ЕАЭС N RU Д-AE.РА01.В.15416/21,2021-06-08
4454,4,N/A,NaN,NaN,ЕАЭС N RU Д-AE.РА01.В.15416/21,2021-06-08


In [107]:
example_numbers = same_tg_low_codes[:5]

truth_unique.loc[
    truth_unique["rd_number"].isin(example_numbers)
].sort_values("rd_number")

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
400,4,N/A,3303,3303009000,ЕАЭС KZ 7500361.13.12.15276,2023-04-13
4121,4,N/A,NaN,NaN,ЕАЭС KZ 7500361.13.12.15276,2023-04-13
1974,4,N/A,3303,3303009000,ЕАЭС N RU Д-AE.ПК08.В.01718/20,2020-11-03
4359,4,N/A,NaN,NaN,ЕАЭС N RU Д-AE.ПК08.В.01718/20,2020-11-03
104,4,N/A,3303,3303001000,ЕАЭС N RU Д-AE.РА01.В.15416/21,2021-06-08
4454,4,N/A,NaN,NaN,ЕАЭС N RU Д-AE.РА01.В.15416/21,2021-06-08
2521,4,N/A,3303,3303009000,ЕАЭС N RU Д-AE.РА01.В.43979/24,2024-01-26
4292,4,N/A,NaN,NaN,ЕАЭС N RU Д-AE.РА01.В.43979/24,2024-01-26
1317,4,N/A,3303,3303001000,ЕАЭС N RU Д-AE.РА01.В.54968/24,2024-01-30
4293,4,N/A,NaN,NaN,ЕАЭС N RU Д-AE.РА01.В.54968/24,2024-01-30


In [108]:
truth_doc_check = (
    truth_unique
    .groupby("rd_number")
    .agg(
        tg_nunique=("tg", "nunique"),
        rd_type_nunique=("rd_type", "nunique"),
        group_tnved_nunique=("group_tnved", "nunique"),
        code_tnved_nunique=("code_tnved", "nunique"),
        rd_date_nunique=("rd_date", "nunique"),
    )
)

print(
    "rd_type conflicts:",
    (truth_doc_check["rd_type_nunique"] > 1).sum()
)

print(
    "rd_date conflicts:",
    (truth_doc_check["rd_date_nunique"] > 1).sum()
)

rd_type conflicts: 37
rd_date conflicts: 14


In [109]:
truth_doc_check[
    (truth_doc_check["rd_type_nunique"] > 1) |
    (truth_doc_check["rd_date_nunique"] > 1)
].head(20)

,tg_nunique,rd_type_nunique,group_tnved_nunique,code_tnved_nunique,rd_date_nunique
rd_number,,,,,
АЭРОЗОЛЬНАЯ УПАКОВКА,1,1,1,3,2
БОЧКА,1,1,4,4,2
ЕАЭС N RU Д-AE.РА02.В.43229/24,2,2,2,2,1
ЕАЭС N RU Д-AE.РА05.В.40313/22,2,2,2,2,1
ЕАЭС N RU Д-CH.РА01.В.10612/21,2,2,1,1,1
ЕАЭС N RU Д-CN.РА06.В.49577/23,2,2,2,2,1
ЕАЭС N RU Д-DE.РА01.В.12875/21,1,1,1,1,2
ЕАЭС N RU Д-DE.РА01.В.25699/21,2,2,2,2,1
ЕАЭС N RU Д-DE.РА02.В.19711/22,2,2,2,2,1


In [110]:
rd_type_conflicts = (
    truth_unique
    .groupby("rd_number")["rd_type"]
    .agg(lambda x: tuple(sorted(set(x))))
)

rd_type_conflicts = rd_type_conflicts[
    rd_type_conflicts.map(len) > 1
]

rd_type_conflicts.value_counts()

rd_type
(N/A, ДС)    33
(ДС, СС)      4
Name: count, dtype: int64

In [111]:
rd_date_conflicts = (
    truth_unique
    .groupby("rd_number")["rd_date"]
    .agg(lambda x: tuple(sorted(set(x.dropna()))))
)

rd_date_conflicts = rd_date_conflicts[
    rd_date_conflicts.map(len) > 1
]

rd_date_conflicts.value_counts()

rd_date
(АЛЮМИНИЙ, ЖЕЛЕЗО)                                                                    1
(ЖЕЛЕЗО, МЕТАЛЛ)                                                                      1
(2021-04-14, 2021-06-04)                                                              1
(2021-05-25, 2021-08-17)                                                              1
(2021-06-08, 2021-09-08)                                                              1
(2021-09-01, 2021-09-26)                                                              1
(2021-04-28, 2021-09-07)                                                              1
(2021-03-10, 2021-06-17)                                                              1
(2021-03-19, 2021-08-30)                                                              1
(274864412, 372371312, 94113212)                                                      1
(ПЛАСТМАССА, ПОЛИЭТИЛЕН ВЫСОКОЙ ПЛОТНОСТИ (HDPE), ПОЛИЭТИЛЕНТЕРЕФТАЛАТ (ПЭТ/ПЭТФ))    1
(304988312, 86048812)   

In [112]:
conflict_numbers = list(rd_type_conflicts.index[:5])

truth_unique.loc[
    truth_unique["rd_number"].isin(conflict_numbers)
].sort_values("rd_number")

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
586,4,N/A,3303,3303001000,ЕАЭС N RU Д-AE.РА02.В.43229/24,2024-03-01
338533,35,ДС,3307,3307900008,ЕАЭС N RU Д-AE.РА02.В.43229/24,NaN
218,4,N/A,3303,3303009000,ЕАЭС N RU Д-AE.РА05.В.40313/22,2022-08-08
339540,35,ДС,3307,3307200000,ЕАЭС N RU Д-AE.РА05.В.40313/22,NaN
4416,4,N/A,NaN,NaN,ЕАЭС N RU Д-CH.РА01.В.10612/21,2021-04-14
320935,35,ДС,3304,3304990000,ЕАЭС N RU Д-CH.РА01.В.10612/21,NaN
1816,4,N/A,3303,3303001000,ЕАЭС N RU Д-CN.РА06.В.49577/23,2023-08-20
383230,35,ДС,3304,3304990000,ЕАЭС N RU Д-CN.РА06.В.49577/23,NaN
2134,4,N/A,3303,3303001000,ЕАЭС N RU Д-DE.РА01.В.25699/21,2021-04-21
22715,43,ДС,3403,3403199000,ЕАЭС N RU Д-DE.РА01.В.25699/21,2021-04-21


In [113]:
date_conflict_numbers = list(rd_date_conflicts.index[:5])

truth_unique.loc[
    truth_unique["rd_number"].isin(date_conflict_numbers)
].sort_values("rd_number")

,tg,rd_type,group_tnved,code_tnved,rd_number,rd_date
179786,43,СС,3403,3403199000,АЭРОЗОЛЬНАЯ УПАКОВКА,АЛЮМИНИЙ
179788,43,СС,3403,3403191000,АЭРОЗОЛЬНАЯ УПАКОВКА,ЖЕЛЕЗО
179794,43,СС,3403,3403199000,АЭРОЗОЛЬНАЯ УПАКОВКА,ЖЕЛЕЗО
179807,43,СС,3403,3403990000,АЭРОЗОЛЬНАЯ УПАКОВКА,ЖЕЛЕЗО
33307,43,СС,3403,3403990000,БОЧКА,ЖЕЛЕЗО
111711,43,СС,2710,2710198200,БОЧКА,МЕТАЛЛ
195389,43,СС,API SL/CF MB 229.1,VW 501.00/505.00,БОЧКА,ЖЕЛЕЗО
207153,43,СС,API SN/SM/CF MB 229.3,VW 502.00/505.00,БОЧКА,ЖЕЛЕЗО
13254,43,ДС,3819,3819000000,ЕАЭС N RU Д-DE.РА01.В.12875/21,2021-04-14
22729,43,ДС,3819,3819000000,ЕАЭС N RU Д-DE.РА01.В.12875/21,2021-06-04


In [114]:
case_numbers = set(case_rank1["rd_documentnumber"])

print(
    "rd_type conflicts matched:",
    sum(x in case_numbers for x in rd_type_conflicts.index)
)

print(
    "rd_date conflicts matched:",
    sum(x in case_numbers for x in rd_date_conflicts.index)
)

rd_type conflicts matched: 37
rd_date conflicts matched: 7


In [115]:
truth_matched = truth_unique[
    truth_unique["rd_number"].isin(case_numbers)
]

print(
    "Matched Truth:",
    len(truth_matched)
)

Matched Truth: 93252


In [116]:
date_like = (
    truth_unique["rd_date"]
    .astype("string")
    .str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)
)

print("Похожи на дату:", date_like.sum())
print("Не похожи на дату:",
      truth_unique["rd_date"].notna().sum() - date_like.sum())

Похожи на дату: 23165
Не похожи на дату: 2401


In [117]:
truth_unique.loc[
    truth_unique["rd_date"].notna() & ~date_like,
    "rd_date"
].value_counts().head(30)

rd_date
ЕАЭС N RU Д-GB.РА08.В.16237/24                                                                                         20
2022-12-19 ||| 2025-11-13                                                                                               9
2025-11-13 ||| 2022-12-19                                                                                               9
ПОЛИЭТИЛЕН ВЫСОКОЙ ПЛОТНОСТИ (HDPE)                                                                                     8
КАНИСТРА                                                                                                                7
ЖЕЛЕЗО                                                                                                                  6
ПЛАСТМАССА                                                                                                              6
2024-10-16 ||| 2024-09-12                                                                                               5
2022-10-24 ||| 2

In [118]:
truth_unique["rd_number"].isna().sum()

np.int64(0)

In [119]:
truth_unique["rd_number"].str.len().describe()

count    105159.000000
mean         32.534248
std          31.460133
min           1.000000
25%          30.000000
50%          30.000000
75%          30.000000
max        7546.000000
Name: rd_number, dtype: float64

In [120]:
truth_unique.loc[
    truth_unique["rd_number"].str.len() <= 10,
    "rd_number"
].value_counts().head(30)

rd_number
249060612     20
КАНИСТРА      13
ПЛАСТМАССА    12
nan|||nan     12
ЖЕЛЕЗО         5
БОЧКА          4
л              2
4              2
5              2
1              2
168854412      2
;;;;;;         2
Декларация     1
223712         1
335            1
650            1
ЖЕСТЬ          1
262360712      1
2710198200     1
208            1
20             1
ВЕДРО          1
18.9           1
18             1
200            1
5.4            1
210            1
640            1
251913712      1
290891012      1
Name: count, dtype: int64

In [121]:
import re

date_pattern = r"\d{4}-\d{2}-\d{2}"

date_counts = (
    truth_unique["rd_date"]
    .astype("string")
    .str.findall(date_pattern)
    .str.len()
)

date_counts.value_counts().sort_index()

rd_date
0        87
1     23197
2      1563
3       274
4        97
5        63
6        38
7        44
8        55
9        29
10       16
11       96
12        6
13        1
Name: count, dtype: int64

In [122]:
truth_unique.loc[
    (truth_unique["rd_date"].notna()) &
    (date_counts == 0),
    "rd_date"
].value_counts().head(30)

rd_date
ЕАЭС N RU Д-GB.РА08.В.16237/24             20
ПОЛИЭТИЛЕН ВЫСОКОЙ ПЛОТНОСТИ (HDPE)         8
КАНИСТРА                                    7
ЖЕЛЕЗО                                      6
ПЛАСТМАССА                                  6
л                                           3
ВЕДРО                                       3
АЭРОЗОЛЬНАЯ УПАКОВКА                        2
БОЧКА                                       2
ЕАЭС № BY/112 11.01. ТР030 028.01 01204     2
51512                                       1
ЕАЭС N RU Д-FR.РА02.В.16527/23              1
86048812                                    1
ЕАЭС N RU Д-JP.РА08.В.94524/22              1
4                                           1
SHELL                                       1
274864412                                   1
372371312                                   1
nan ||| nan                                 1
293301012                                   1
АЭРОЗОЛЬНАЯ УПАКОВКА ||| КОРОБКА/БОКС       1
ПОЛИЭТИЛЕНТЕРЕФТАЛАТ (ПЭТ/

In [123]:
truth_unique["is_matched"] = (
    truth_unique["rd_number"].isin(case_numbers)
)

truth_unique["rd_len"] = truth_unique["rd_number"].str.len()

In [124]:
truth_unique.groupby("is_matched")["rd_len"].describe()

,count,mean,std,min,25%,50%,75%,max
is_matched,,,,,,,,
False,11907.0,53.089863,90.853682,1.0,30.0,30.0,63.0,7546.0
True,93252.0,29.909578,1.175187,19.0,30.0,30.0,30.0,39.0


In [125]:
truth_unique.loc[
    ~truth_unique["is_matched"],
    "rd_number"
].value_counts().head(30)

rd_number
249060612                                                                                                                                                                                                                                                                              20
КАНИСТРА                                                                                                                                                                                                                                                                               13
ПЛАСТМАССА                                                                                                                                                                                                                                                                             12
nan|||nan                                                                                                                                       

In [126]:
coverage_by_tg = (
    truth_unique
    .groupby("tg")["is_matched"]
    .agg(
        total="count",
        matched="sum",
        coverage="mean"
    )
)

coverage_by_tg

,total,matched,coverage
tg,,,
35,79303,70132,0.884355
4,4455,4449,0.998653
43,21401,18671,0.872436


In [127]:
matched_truth = truth_unique[truth_unique["is_matched"]]

matched_docs_by_tg = (
    matched_truth
    .groupby("tg")["rd_number"]
    .nunique()
)

matched_docs_by_tg

tg
35    68370
4      3936
43    14061
Name: rd_number, dtype: int64

In [132]:
total_docs_by_tg = (
    truth_unique
    .groupby("tg")["rd_number"]
    .nunique()
)

pd.DataFrame({
    "total_docs": total_docs_by_tg,
    "matched_docs": matched_docs_by_tg
}).assign(
    coverage=lambda x: round((x["matched_docs"] / x["total_docs"]) * 100, 3)
)

,total_docs,matched_docs,coverage
tg,,,
35,77416,68370,88.315
4,3942,3936,99.848
43,16540,14061,85.012


In [133]:
coverage_rd_type = (
    truth_unique
    .groupby(["tg", "rd_type"])["is_matched"]
    .agg(
        total="count",
        matched="sum",
        coverage="mean"
    )
)

coverage_rd_type

total  matched  coverage
tg rd_type                          
35 ДС       60054    58310  0.970959
   СГР      18759    11546  0.615491
   СС         490      276  0.563265
4  N/A       4455     4449  0.998653
43 ДС       21012    18493  0.880116
   СГР        269      142  0.527881
   СС         120       36  0.300000

In [134]:
doc_level_truth = (
    truth_unique
    .groupby("rd_number")
    .agg(
        tg=("tg", lambda x: tuple(sorted(set(x)))),
        rd_type=("rd_type", lambda x: tuple(sorted(set(x)))),
        is_matched=("is_matched", "max")
    )
    .reset_index()
)

In [135]:
doc_level_truth.explode("tg").groupby(["tg", "is_matched"]).size()

tg  is_matched
35  False          9046
    True          68370
4   False             6
    True           3936
43  False          2479
    True          14061
dtype: int64

In [136]:
doc_tg_type = (
    truth_unique[
        ["rd_number", "tg", "rd_type", "is_matched"]
    ]
    .drop_duplicates()
)

In [137]:
doc_tg_type.groupby(
    ["tg", "rd_type"]
)["is_matched"].agg(
    total_docs="count",
    matched_docs="sum",
    coverage="mean"
)

total_docs  matched_docs  coverage
tg rd_type                                    
35 ДС            58563         56841  0.970596
   СГР           18382         11262  0.612665
   СС              475           271  0.570526
4  N/A            3942          3936  0.998478
43 ДС            16230         13913  0.857240
   СГР             251           126  0.501992
   СС               59            22  0.372881

In [138]:
base_truth = (
    truth_unique
    .groupby("rd_number")
    .agg(
        tg=("tg", lambda x: sorted(set(x))),
        rd_type=("rd_type", lambda x: sorted(set(x))),
        tnved_codes=("code_tnved", lambda x: sorted(set(x.dropna()))),
    )
    .reset_index()
)

In [140]:
print(base_truth.shape)
base_truth.head()

(97822, 4)


,rd_number,tg,rd_type,tnved_codes
0,- Блеск для губ BOURJOIS PARIS GLOSS FABULEU...,[35],[СС],[3304100000]
1,- Водостойкая Тушь для ресниц MAX FACTOR MAS...,[35],[СС],[3304200000]
2,- Консилер RIMMEL LONDON THE MULTI-TASKER CO...,[35],[СС],[3304990000]
3,- Лак для ногтей MAX FACTOR X MASTERPIECE XP...,[35],[СС],[3304300000]
4,- Лак для ногтей RIMMEL LONDON HOLOGRAPHIC T...,[35],[СС],[3304300000]


In [141]:
base_truth["tg"].value_counts()

tg
[35]        77341
[43]        16496
[4]          3909
[35, 43]       43
[35, 4]        32
[4, 43]         1
Name: count, dtype: int64

In [142]:
base_df = case_rank1.merge(
    base_truth,
    left_on="rd_documentnumber",
    right_on="rd_number",
    how="inner"
)

In [143]:
print("base_df.shape:", base_df.shape)
print(
    "unique RD:",
    base_df["rd_documentnumber"].nunique()
)
print(
    "duplicate RD:",
    base_df["rd_documentnumber"].duplicated().sum()
)

base_df.shape: (86301, 8)
unique RD: 86301
duplicate RD: 0


In [144]:
base_df["tg"].value_counts()

tg
[35]        68305
[43]        14027
[4]          3903
[35, 43]       33
[35, 4]        32
[4, 43]         1
Name: count, dtype: int64

In [145]:
print(
    "case rank1 docs:",
    case_rank1["rd_documentnumber"].nunique()
)

print(
    "base_df docs:",
    base_df["rd_documentnumber"].nunique()
)

case rank1 docs: 3595803
base_df docs: 86301


*EDA завершён.*

Мы сформировали документный слой RD через rank == 1 и агрегированный Truth по rd_number. Подтверждено наличие multi-label и нескольких товарных позиций внутри одного РД. Coverage сопоставления неоднороден по TG и типам РД, поэтому unmatched-документы не используются как эталонная разметка.

Следующий этап — Feature Discovery: исследование структурированных полей и текстовых характеристик RD.